# NB11 — PANDA Gleason Grading (Multi-Head Attention MIL)

Train a multiple-instance-learning model on PANDA features from NB10 to predict ISUP grade (0–5). Per-scale multi-head attention pooling, fused 2-layer MLP, focal cross-entropy with label smoothing and ordinal/expectation regularizers, AdamW + cosine schedule + EMA. 5-fold stratified cross-validation across 3 seeds (42, 777, 1337); seed-averaged out-of-fold (OOF) probabilities form the final ensemble.

Token budgets match manuscript Section 2B: 1,200 tiles at 0.5 μm/pixel + 400 at 2.0 μm/pixel = 1,600 tokens per slide. Reports OOF quadratic-weighted κ and accuracy. Saves `oof_ensemble.csv` (image_id, true ISUP, predicted ISUP, per-class probabilities, data_provider) for NB12 and NB13.

In [ ]:
import os, sys, json, time, math, platform, random
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

WORKSPACE  = Path(os.environ.get('WORKSPACE',  './workspace'))
PANDA_ROOT = Path(os.environ.get('PANDA_ROOT', './data/PANDA'))
FEAT_05    = WORKSPACE / 'features' / 'panda' / 'scale0p5'
FEAT_20    = WORKSPACE / 'features' / 'panda' / 'scale2p0'
OUTPUT     = WORKSPACE / 'results' / 'panda_mil'
(OUTPUT / 'models').mkdir(parents=True, exist_ok=True)
(OUTPUT / 'logs').mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'

@dataclass
class CFG:
    INPUT_DIM: int = 768
    NUM_CLASSES: int = 6
    NUM_HEADS: int = 8
    POOL_DROPOUT: float = 0.1
    TOPK_RATIO: float = 0.15
    TOPK_MIN: int = 24
    HIDDEN_DIM: int = 512
    FUSE_DROPOUT: float = 0.15
    EPOCHS: int = 30
    WARMUP_EPOCHS: int = 2
    BATCH_SLIDES: int = 24
    LR: float = 3e-4
    WD: float = 1e-4
    MAX_GRAD_NORM: float = 1.0
    AMP: bool = True
    NUM_WORKERS: int = 0 if platform.system() == 'Windows' else max(4, (os.cpu_count() or 8) - 2)
    PIN_MEMORY: bool = torch.cuda.is_available()
    PROVIDER_AWARE: bool = True
    PATIENCE: int = 7
    LABEL_SMOOTH: float = 0.05
    FOCAL_ALPHA: float = 0.25
    FOCAL_GAMMA: float = 2.0
    ORDINAL_LAM: float = 0.25
    EXP_LAM: float = 0.02
    NOISE_STD: float = 0.01
    FEAT_DROPOUT_P: float = 0.05
    MAX_TILES_05: int = 1200
    MAX_TILES_20: int = 400
    N_FOLDS: int = 5
    SEEDS: Tuple[int, ...] = (42, 777, 1337)
    EMA_DECAY: float = 0.999

C = CFG()

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

class PANDAIndex:
    def __init__(self, cfg):
        df = pd.read_csv(PANDA_ROOT / 'train.csv')
        df['isup_grade'] = df['isup_grade'].fillna(0).astype(int)
        has05 = df['image_id'].apply(lambda s: (FEAT_05 / f'{s}.npy').exists())
        has20 = df['image_id'].apply(lambda s: (FEAT_20 / f'{s}.npy').exists())
        df = df[has05 & has20].reset_index(drop=True)
        self.df = df
        print(f'[DATA] slides with both scales: {len(df)}')
    def splits(self):
        n = len(self.df)
        if C.PROVIDER_AWARE and 'data_provider' in self.df.columns:
            key = self.df['isup_grade'].astype(str) + '_' + self.df['data_provider'].astype(str)
        else:
            key = self.df['isup_grade']
        skf = StratifiedKFold(n_splits=C.N_FOLDS, shuffle=True, random_state=42)
        return list(skf.split(np.arange(n), key))

class SlideBagDataset(Dataset):
    def __init__(self, df, train):
        self.df = df.reset_index(drop=True)
        self.train = train
    def __len__(self): return len(self.df)
    def _load(self, root, sid, budget, train):
        arr = np.load(root / f'{sid}.npy', mmap_mode='r')
        if arr.ndim != 2 or arr.shape[1] != C.INPUT_DIM:
            arr = np.asarray(arr, dtype=np.float32).reshape(-1, C.INPUT_DIM)
        T = arr.shape[0]
        if T == 0:
            arr = np.zeros((1, C.INPUT_DIM), dtype=np.float32); T = 1
        if T > budget:
            if train:
                idx = np.random.choice(T, budget, replace=False); arr = arr[idx]
            else:
                arr = arr[:budget]
        return np.asarray(arr, dtype=np.float32).copy(order='C')
    def __getitem__(self, i):
        r = self.df.iloc[i]; sid = r['image_id']; y = int(r['isup_grade'])
        f05 = self._load(FEAT_05, sid, C.MAX_TILES_05, self.train)
        f20 = self._load(FEAT_20, sid, C.MAX_TILES_20, self.train)
        if self.train:
            if np.random.rand() < 0.4:
                f05 = f05 + np.random.normal(0, C.NOISE_STD, f05.shape).astype(np.float32)
                f20 = f20 + np.random.normal(0, C.NOISE_STD, f20.shape).astype(np.float32)
            if np.random.rand() < 0.3:
                f05 *= (np.random.rand(*f05.shape) > C.FEAT_DROPOUT_P).astype(np.float32)
                f20 *= (np.random.rand(*f20.shape) > C.FEAT_DROPOUT_P).astype(np.float32)
        return {
            'feat_05': torch.from_numpy(f05),
            'feat_20': torch.from_numpy(f20),
            'label': torch.tensor(y, dtype=torch.long),
            'id': sid,
            'prov': r.get('data_provider', 'NA'),
        }

def collate_bags(batch):
    B = len(batch); D = batch[0]['feat_05'].shape[1]
    n1 = [b['feat_05'].shape[0] for b in batch]
    n2 = [b['feat_20'].shape[0] for b in batch]
    N1, N2 = max(n1), max(n2)
    f05 = torch.zeros(B, N1, D); f20 = torch.zeros(B, N2, D)
    m05 = torch.ones(B, N1, dtype=torch.bool); m20 = torch.ones(B, N2, dtype=torch.bool)
    for i, b in enumerate(batch):
        a = b['feat_05']; f05[i, :a.size(0)] = a; m05[i, :a.size(0)] = False
        c = b['feat_20']; f20[i, :c.size(0)] = c; m20[i, :c.size(0)] = False
    y = torch.stack([b['label'] for b in batch], 0)
    return {
        'feat_05': f05, 'mask_05': m05,
        'feat_20': f20, 'mask_20': m20,
        'label': y,
        'ids':  [b['id'] for b in batch],
        'prov': [b['prov'] for b in batch],
    }

class MultiHeadPool(nn.Module):
    def __init__(self, d_model, num_heads, pdrop):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=pdrop, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.ReLU(), nn.Dropout(pdrop),
            nn.Linear(d_model*2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(pdrop)
        self.scorer = nn.Sequential(nn.Linear(d_model, d_model//2), nn.ReLU(), nn.Linear(d_model//2, 1))
    def forward(self, x, mask):
        x = x.float()
        attn_out, _ = self.attn(x, x, x, key_padding_mask=mask, need_weights=False)
        h = self.norm1(x + self.dropout(attn_out))
        f = self.ffn(h)
        h = self.norm2(h + self.dropout(f))
        scores = self.scorer(h).squeeze(-1).masked_fill(mask, -1e4)
        with torch.no_grad():
            nonpad = (~mask).sum(1)
            k = (nonpad.float() * C.TOPK_RATIO).floor().clamp(min=C.TOPK_MIN)
            k = torch.minimum(k, nonpad.float()).to(torch.long).clamp(min=1)
        pooled = []
        for b in range(x.size(0)):
            k_b = int(k[b].item())
            topv, topi = torch.topk(scores[b, :nonpad[b]], k_b, dim=0)
            sel = h[b, topi]
            w = F.softmax(topv, dim=0).unsqueeze(1)
            pooled.append((sel * w).sum(0))
        return torch.stack(pooled, dim=0)

class MILModel(nn.Module):
    def __init__(self, in_dim=C.INPUT_DIM, num_classes=C.NUM_CLASSES):
        super().__init__()
        self.pool05 = MultiHeadPool(in_dim, C.NUM_HEADS, C.POOL_DROPOUT)
        self.pool20 = MultiHeadPool(in_dim, C.NUM_HEADS, C.POOL_DROPOUT)
        self.fuse = nn.Sequential(
            nn.Linear(in_dim*2, C.HIDDEN_DIM), nn.LayerNorm(C.HIDDEN_DIM),
            nn.ReLU(), nn.Dropout(C.FUSE_DROPOUT),
            nn.Linear(C.HIDDEN_DIM, C.HIDDEN_DIM), nn.ReLU(), nn.Dropout(C.FUSE_DROPOUT))
        self.classifier = nn.Linear(C.HIDDEN_DIM, num_classes)
    def forward(self, f05, m05, f20, m20):
        e05 = self.pool05(f05, m05)
        e20 = self.pool20(f20, m20)
        h = self.fuse(torch.cat([e05, e20], dim=1))
        return {'logits': self.classifier(h).float(), 'emb': h}

def smooth_one_hot(y, num_classes, eps):
    with torch.no_grad():
        target = torch.full((y.size(0), num_classes), eps/num_classes, device=y.device)
        target.scatter_(1, y.unsqueeze(1), 1.0 - eps + eps/num_classes)
    return target

def focal_ce_loss(logits, targets, class_weights=None, alpha=0.25, gamma=2.0, label_smooth=0.05):
    K = logits.size(1)
    logp = F.log_softmax(logits, dim=1)
    p = logp.exp()
    T = smooth_one_hot(targets, K, label_smooth)
    pt = (p * T).sum(dim=1).clamp(min=1e-8)
    ce = -(T * logp).sum(dim=1)
    focal = (alpha * (1 - pt)**gamma) * ce
    if class_weights is not None:
        focal = focal * class_weights[targets]
    return focal.mean()

def ordinal_penalties(probs, targets):
    pred = probs.argmax(dim=1)
    dist = (pred - targets).abs().float()
    expc = (probs * torch.arange(C.NUM_CLASSES, device=probs.device).float()).sum(dim=1)
    return dist.mean(), ((expc - targets.float())**2).mean()

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)
    def apply_shadow(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])
        self.backup = {}

def compute_class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    freq = counts / np.clip(counts.sum(), 1, None)
    inv = 1.0 / np.clip(freq, 1e-6, None)
    inv = inv / inv.mean()
    return torch.tensor(inv, dtype=torch.float32)

class Trainer:
    def __init__(self, in_dim, fold, seed, class_weights):
        self.fold = fold; self.seed = seed
        self.model = MILModel(in_dim, C.NUM_CLASSES).to(DEVICE)
        self.ema = EMA(self.model, decay=C.EMA_DECAY)
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=C.LR, weight_decay=C.WD)
        self.scaler = torch.amp.GradScaler('cuda', enabled=C.AMP)
        self.best_kappa = -1.0; self.no_improve = 0
        self.cw = class_weights.to(DEVICE) if class_weights is not None else None
        self.lr_warm = torch.optim.lr_scheduler.LinearLR(self.opt, start_factor=0.1, total_iters=C.WARMUP_EPOCHS)
        self.lr_main = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.opt, T_max=max(1, C.EPOCHS - C.WARMUP_EPOCHS), eta_min=1e-6)
        self.ckpt = OUTPUT / 'models' / f'seed{seed}_fold{fold}_best.pth'
    def run_epoch(self, loader, train):
        self.model.train(train)
        total = 0.0; y_true = []; y_pred = []
        for b in loader:
            f05 = b['feat_05'].to(DEVICE); f20 = b['feat_20'].to(DEVICE)
            m05 = b['mask_05'].to(DEVICE); m20 = b['mask_20'].to(DEVICE)
            y = b['label'].to(DEVICE)
            if train:
                with torch.amp.autocast(device_type='cuda', enabled=C.AMP):
                    out = self.model(f05, m05, f20, m20); logits = out['logits']
                    loss_ce = focal_ce_loss(logits, y, self.cw)
                    probs = F.softmax(logits.float(), dim=1)
                    dist_mean, mse_mean = ordinal_penalties(probs, y)
                    loss = loss_ce + C.ORDINAL_LAM * dist_mean + C.EXP_LAM * mse_mean
                self.opt.zero_grad(set_to_none=True)
                self.scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), C.MAX_GRAD_NORM)
                self.scaler.step(self.opt); self.scaler.update()
                self.ema.update(self.model)
            else:
                self.ema.apply_shadow(self.model)
                with torch.no_grad(), torch.amp.autocast(device_type='cuda', enabled=C.AMP):
                    out = self.model(f05, m05, f20, m20); logits = out['logits']
                    loss_ce = focal_ce_loss(logits, y, self.cw, label_smooth=0.0)
                    probs = F.softmax(logits.float(), dim=1)
                    dist_mean, mse_mean = ordinal_penalties(probs, y)
                    loss = loss_ce + C.ORDINAL_LAM * dist_mean + C.EXP_LAM * mse_mean
                self.ema.restore(self.model)
            total += float(loss.item())
            pred = logits.detach().float().softmax(1).argmax(1)
            y_true.extend(y.tolist()); y_pred.extend(pred.tolist())
        avg = total / len(loader)
        return avg, qwk(np.array(y_true), np.array(y_pred)), accuracy_score(np.array(y_true), np.array(y_pred))
    def fit(self, dl_tr, dl_va):
        for ep in range(1, C.EPOCHS+1):
            trL, trK, _ = self.run_epoch(dl_tr, train=True)
            vaL, vaK, _ = self.run_epoch(dl_va, train=False)
            if ep <= C.WARMUP_EPOCHS: self.lr_warm.step()
            else: self.lr_main.step()
            print(f'  E{ep:02d}  train L={trL:.4f} k={trK:.4f}  |  val L={vaL:.4f} k={vaK:.4f}')
            if vaK > self.best_kappa:
                self.best_kappa = vaK; self.no_improve = 0
                torch.save({'model': self.model.state_dict(), 'ema': self.ema.shadow, 'kappa': vaK}, self.ckpt)
            else:
                self.no_improve += 1
                if self.no_improve >= C.PATIENCE:
                    print('  early stopping'); break
    @torch.no_grad()
    def predict_val(self, loader):
        if self.ckpt.exists():
            ck = torch.load(self.ckpt, map_location=DEVICE)
            self.model.load_state_dict(ck['model'])
            self.ema.shadow = ck.get('ema', self.ema.shadow)
        self.model.eval(); self.ema.apply_shadow(self.model)
        preds = []; probs = []; labels = []
        for b in loader:
            f05 = b['feat_05'].to(DEVICE); f20 = b['feat_20'].to(DEVICE)
            m05 = b['mask_05'].to(DEVICE); m20 = b['mask_20'].to(DEVICE)
            out = self.model(f05, m05, f20, m20)
            p = F.softmax(out['logits'].float(), dim=1)
            probs.append(p.cpu().numpy())
            preds.append(p.argmax(1).cpu().numpy())
            labels.extend(b['label'].tolist())
        self.ema.restore(self.model)
        return {'preds': np.concatenate(preds, 0), 'probs': np.concatenate(probs, 0), 'labels': np.array(labels)}

print(f'PANDA MIL training: device={DEVICE}, gpu={GPU_NAME}')
print(f'  budgets: {C.MAX_TILES_05} @ 0.5 μm/px + {C.MAX_TILES_20} @ 2.0 μm/px')
idx = PANDAIndex(C); splits = idx.splits()
N = len(idx.df); y_all = idx.df['isup_grade'].values
oof_prob_ens = np.zeros((N, C.NUM_CLASSES), dtype=np.float32)

for seed in C.SEEDS:
    set_seed(seed)
    print(f'\nseed {seed}')
    oof_prob = np.zeros((N, C.NUM_CLASSES), dtype=np.float32)
    for f, (tr, va) in enumerate(splits, start=1):
        print(f' fold {f}/{C.N_FOLDS}')
        df_tr = idx.df.iloc[tr].reset_index(drop=True)
        df_va = idx.df.iloc[va].reset_index(drop=True)
        cw = compute_class_weights(df_tr['isup_grade'].values, C.NUM_CLASSES)
        ds_tr = SlideBagDataset(df_tr, train=True)
        ds_va = SlideBagDataset(df_va, train=False)
        dl_tr = DataLoader(ds_tr, batch_size=C.BATCH_SLIDES, shuffle=True,
                           num_workers=C.NUM_WORKERS, pin_memory=C.PIN_MEMORY, collate_fn=collate_bags)
        dl_va = DataLoader(ds_va, batch_size=C.BATCH_SLIDES, shuffle=False,
                           num_workers=C.NUM_WORKERS, pin_memory=C.PIN_MEMORY, collate_fn=collate_bags)
        trainer = Trainer(C.INPUT_DIM, f, seed, cw)
        trainer.fit(dl_tr, dl_va)
        out = trainer.predict_val(dl_va)
        oof_prob[va] = out['probs']
        kappa = qwk(out['labels'], out['preds'])
        print(f'  seed {seed} fold {f}: κ={kappa:.4f}')
        del ds_tr, ds_va, dl_tr, dl_va, trainer
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    oof_prob_ens += oof_prob / float(len(C.SEEDS))

oof_pred_ens = oof_prob_ens.argmax(1)
kappa_oof = qwk(y_all, oof_pred_ens)
acc_oof = accuracy_score(y_all, oof_pred_ens)
print(f'\nOOF ensemble: κ={kappa_oof:.4f}  acc={acc_oof:.4f}')

df_out = pd.DataFrame({
    'image_id': idx.df['image_id'],
    'true_isup': y_all,
    'pred_isup': oof_pred_ens,
    **{f'prob_{i}': oof_prob_ens[:, i] for i in range(C.NUM_CLASSES)},
})
if 'data_provider' in idx.df.columns:
    df_out['data_provider'] = idx.df['data_provider'].values
df_out.to_csv(OUTPUT / 'oof_ensemble.csv', index=False)

with open(OUTPUT / 'summary.json', 'w') as f:
    json.dump({
        'oof_kappa': float(kappa_oof),
        'oof_accuracy': float(acc_oof),
        'seeds': list(C.SEEDS),
        'budgets': {'0p5': C.MAX_TILES_05, '2p0': C.MAX_TILES_20},
    }, f, indent=2)

print(f'[DONE] NB11 complete. OOF predictions: {OUTPUT / "oof_ensemble.csv"}.')